In [73]:
# import necessary libraries
import pandas as pd
import numpy as np
from pathlib import  Path
from sklearn.metrics import mean_absolute_error


In [74]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"

In [75]:
#load the demand and forecast datasets from parquet files
data = pd.read_parquet(DATA_DIR / "gb_demand_features.parquet")

In [76]:
data.columns

Index(['SETTLEMENT_DATE', 'SETTLEMENT_PERIOD', 'ND', 'is_imputed',
       'datetime_local', 'local_date', 'local_time',
       'elexon_forecast_published_utc', 'elexon_forecast', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'cloud_cover',
       'wind_speed_10m', 'shortwave_radiation', 'forecast_temperature_2m',
       'forecast_relative_humidity_2m', 'forecast_precipitation',
       'forecast_cloud_cover', 'forecast_wind_speed_10m',
       'forecast_shortwave_radiation', 'year', 'month', 'day_of_week', 'hour',
       'minute', 'day_of_year', 'quarter', 'is_weekend', 'is_holiday',
       'lag_2d', 'xmas_period', 'day_before_holiday', 'day_after_holiday',
       'lag_7d', 'lag_14d', 'lag_21d', 'lag_28d', 'lag_35d', 'lag_42d',
       'prev_6weeks_lag_noholiday', 'lag7d_minus_prev_6weeks',
       'seasonal_naive'],
      dtype='str')

In [77]:
columns =[ 'local_date', 'local_time','ND','seasonal_naive','elexon_forecast']

In [78]:
df = data[columns].dropna(subset=["elexon_forecast"]).copy()

In [79]:
df.head(10)

,local_date,local_time,ND,seasonal_naive,elexon_forecast
datetime,,,,,
2021-04-27 23:00:00+00:00,2021-04-28,00:00,22321.0,23025.0,22720.0
2021-04-27 23:30:00+00:00,2021-04-28,00:30,21983.0,22392.0,22120.0
2021-04-28 00:00:00+00:00,2021-04-28,01:00,22172.0,22444.0,22192.0
2021-04-28 00:30:00+00:00,2021-04-28,01:30,22546.0,22968.0,22610.0
2021-04-28 01:00:00+00:00,2021-04-28,02:00,22582.0,22847.0,22378.0
2021-04-28 01:30:00+00:00,2021-04-28,02:30,22177.0,22589.0,22059.0
2021-04-28 02:00:00+00:00,2021-04-28,03:00,21883.0,22029.0,21738.0
2021-04-28 02:30:00+00:00,2021-04-28,03:30,21798.0,21782.0,21379.0
2021-04-28 03:00:00+00:00,2021-04-28,04:00,21420.0,21610.0,21080.0


In [80]:
df.isnull().sum()

local_date         0
local_time         0
ND                 0
seasonal_naive     0
elexon_forecast    0
dtype: int64

In [81]:
mae = mean_absolute_error(df['ND'], df['seasonal_naive'])
print(mae)

2027.9426715234626


In [82]:
df['elexon_abs_error_vs_nd'] = np.abs(df['elexon_forecast'] - df['ND'])
df['seasonal_naive_abs_error'] = np.abs(df['seasonal_naive'] - df['ND'])

In [83]:
seasonal_mae = df['seasonal_naive_abs_error'].mean()
elexon_vs_nd_mae = df['elexon_abs_error_vs_nd'].mean()

print(f"Seasonal Naive MAE: {seasonal_mae:.2f} MW")
print(f"Elexon MAE vs ND:     {elexon_vs_nd_mae:.2f} MW")

Seasonal Naive MAE: 2027.94 MW
Elexon MAE vs ND:     658.04 MW


In [84]:
df.groupby(
    df['local_date'].dt.year
)[
    ['seasonal_naive_abs_error',
     'elexon_abs_error_vs_nd']
].mean()

,seasonal_naive_abs_error,elexon_abs_error_vs_nd
local_date,,
2021,1635.587584,555.281445
2022,2010.491491,631.313543
2023,2040.403745,607.735256
2024,2008.580466,628.765084
2025,2176.766731,711.185791
2026,2110.959521,781.636401
